# Concrete Mix Calculations

- Manually design concrete mix based on target specification and assumed parameters.
- Input the calculated material quantities into the model and compare predicted strength to the one specified in the calculations.
- The examples only covers the conventional materials in designing a mix. No SCMs or superplasticizer are consider.
    - **Cement**
    - **Sand** (Fine aggregate)
    - **Gravel** (Coarse aggregate)
- The method of calculating the design mix would be based on ACI (using ACI PRC-211.1-22). The standard design mix procedure can be summarized as follows.
    1. Choice of slump
    2. Select maximum size of aggregate
    3. Select mixing water amount and air content
    4. Select water-cement ratio and proportion of SCMs
    5. Determine cement and SCM content
    6. Determine coarse aggregate content
    7. Determine fine aggregate content
    8. Make adjustment for aggregate moisture

In [1]:
import math
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

In [2]:
parent_dir = Path().cwd()
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

model_path = parent_dir / "csp_model.joblib"
csp_model_pipeline = joblib.load(model_path)
csp_model_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('setup_format', ...), ('setup_features', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['cement','blast_furnace_slag','fly_ash',...,'water','superplasticizer', 'age']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function set...x7dea2b319900>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False


### Situation 1

*Modified example from ACI PRC-211.1-22 (2022)*

Concrete is required for a portion of a structure that will be below ground level in a location where it will not be exposed to severe weathering, constant wetting, or sulfate attack. It is in Class FO. A strength of 3000 psi at 28 days is specified. A locally available rounded coarse aggregate with a nominal maximum size of 1.5 in. is suitable. This coarse aggregate has a saturated surface-dry (SSD) specific gravity of 2.68, absorption (A%) of 0.5%, and a dry-rodded density of 100 lb/ft^3. The fine aggregate has a fineness modulus of 2.80, an SSD specific gravity of 2.64, and an absorption (A%) of 0.7%. The quantities of mixture constituents per cubic meter (m^3) of concrete are determined.

**Summary of key information:**

- Target strength at 28 days: 3000 psi (~21 MPa)
- Cement:
    - SG: 3.15 (typical)
- Coarse aggregate (CA)
    - Nominal max. size: 1.5 in. (~40 mm)
    - SSD SG: 2.68
    - Absorption: 0.5%
    - Dry-rodded density: 100 lb/ft^3 (~1602 kg/m^3)
- Fine aggregate (FA)
    - Fineness modulus (FM): 2.80
    - SSD SG: 2.64
    - Absorption: 0.7%
- Exposure class: F0 -> Non-air-entrained

**Notes:**
- For calculation convenience, we simply get the amount in pound per cubic yards (lb/yd^3) first, then convert them into kilograms per cubic meter (kg/m^3) at the end.
- For simplicity, we would not cover moisture adjustment here. However, actual practice covers adjusting the moisture content based on the condition of the materials.

#### Calculation proper

1. Estimate slump (Table 5.3.1): 3-4 in (75-100 mm)
2. Select nominal max. size of CA: 40mm
3. Estimate water content (Table 5.3.3):
    - Approx. amount of mixing water = 300 lb/yd^3
    - Approx. amount of entrapped air = 1%

In [3]:
# Amount of water
W_water = 300
W_water

300

4. Estimate water-cement ratio, w/c:
    - Historical test data to establish standard deviation is not given, hence we are required to add overdesign factor to the mix.
    - From Table 4.7.4.1, for a target strength of 3000 psi (~21 MPa), it falls between the 3000-5000 psi range (21-35 MPa) of adding 1200 psi (~8.3 MPa).
    - Design the mix for 3000 + 1200 = **4200 psi (~29 MPa)**
    - From Table 5.3.4, we can interpolate the w/c between 4000-5000 psi under non-air-entrained concrete.

In [4]:
fc_range = np.array([4000, 5000])
ratio_wc_range = np.array([0.57, 0.48])

fc_interpolate = 4200
ratio_wc = float(np.interp(fc_interpolate, fc_range, ratio_wc_range).round(4))
ratio_wc

0.552

5. Calculate cement content:
    - Note that water-cement ratio is just the ratio between the amount of cement and the amount of water.

In [5]:
W_cement = math.ceil(W_water / ratio_wc)
W_cement

544

6. Calculate CA content:
    - From Table 5.3.6., given the FM and nominal max. size of CA, we adapt a dry-rodded CA volumne of 0.71 ft^3 per unit vol. of concrete.
    - Recall that the dry-rodded density is 100 lb/ft^3, each bulk cubic foot of CA would weigh 71 lb.
    - Adjust the proportion based on a cubic yard (27 ft^3), then take into account absorption to convert dry-rodded density to the corresponding SSD weight.

In [6]:
W_ca = round((0.71 * 100) * 27 * (1 + 0.5/100))
W_ca

1927

7. Calculate FA content:
    - Concrete consists of water, air, cement, coarse aggregate, and fine aggregate.
    - For the cubic yard being proportioned, the weights of all these except the fine aggregate have been determined.
    - Approach is to get the absolute volume of each component through their weight-volume relationships determined by their corresponding specific gravities.
    - The volume of the fine aggregate required is determined by adding the total volume of all other mixture constituents subtracted from the total volume of one cubic yard.

In [7]:
V_water = W_water / 62.4
V_water

4.8076923076923075

In [8]:
V_cement = W_cement / (3.15 * 62.4)
V_cement

2.7676027676027677

In [9]:
V_ca = W_ca / (2.68 * 62.4)
V_ca

11.522914274779946

In [10]:
V_air = 0.01 * 27
V_air

0.27

In [11]:
V_total = V_water + V_cement + V_ca + V_air
V_total

19.368209350075023

In [12]:
V_fa = 27 - V_total
V_fa

7.631790649924977

In [13]:
# Reqd SSD weight of FA
W_fa = round(V_fa * 2.64 * 62.4)
W_fa

1257

In [14]:
# Checking actual w/c ratio
W_water / W_cement

0.5514705882352942

In [15]:
# lb/yd^3 to kg/m^3
conversion_factor = 0.59328 

W_total = W_cement + W_ca + W_fa + W_water

results = {"lb/yd^3": [W_cement, W_ca, W_fa, W_water, W_total]}
index = ["cement", "coarse_aggregate", "fine_aggregate", "water", "total"]

results_df = pd.DataFrame(results, index=index)
results_df["kg/m^3"] = (results_df * conversion_factor).round()
results_df

,lb/yd^3,kg/m^3
cement,544,323.0
coarse_aggregate,1927,1143.0
fine_aggregate,1257,746.0
water,300,178.0
total,4028,2390.0


Calculated weight falls within the typical weight of concrete -> 2200-2400 kg/m^3

Recall that this mix has a target strength of 29 **MPa**. We can feed the following to the model to get a prediction, then compare the result.

In [16]:
test_input = {
    "cement": 323,
    "blast_furnace_slag": 0,
    "fly_ash": 0,
    "silica_fume": 0,
    "coarse_aggregate": 1143,
    "fine_aggregate": 746,
    "water": 178,
    "superplasticizer": 0,
    "age": 28
}

csp_model_pipeline.predict(test_input)[0]

np.float32(31.87111)

Remarks:

- The model predicts ~32 MPa, higher by about 3 MPa than the required target prescribed by ACI standard. Because the discrepancy lies around the value of mean average error (MAE) of the chosen model (see [model development](../nb_model_dev.ipynb)), we can feel confident in the prediction.
- It's still a best practice to compare prediction to material test results of multiple samples of the same mix.

## Situation 2

For a certain residential building project, we are prescribed to use a mix proportion of 1:2:4 based on a 40 kg bag of cement and use 24 L of water per bag of cement. The bulk unit weight of cement, sand, and gravel were determined on site through conventional means and the specific gravities were assumed based on common literature. The values are summarized as follows. 

| Material | Specific Gravity (SG) | Bulk Unit Weight (kg/m^3) |
| --- | --- | --- |
| Cement | 3.15 | 1506 |
| Sand | 2.65 | 1680 |
| Gravel | 2.50 | 1525 |

- Sand and Gravel are normally classified as fine and coarse aggregate, respectively.
- When reading proportions like 1:2:4, it refers to the amount of cement, sand, and gravel based on volume. In other words, for every bag of cement, we use 2 parts and 4 parts of sand and gravel, respectively.
- The weight of 24 L of water is 24 kg assuming the density is 1000 kg/m^3. Then assuming normal concrete (non-air-entrained) getting the water/cement ratio gives us 0.6, which gives an estimated strength of 25.7 MPa at 28 days from Table 5.3.4.
- Calculate the actual quantities in kg/m^3, then check the predicted strength of this mix.

In [17]:
# Get the loose volume in cu.m of each component
Vb_cement = 40 / 1506
Vb_sand = Vb_cement * 2
Vb_gravel = Vb_cement * 4
Vb_water = 24 / 1000

Vb_cement, Vb_sand, Vb_gravel, Vb_water

(0.02656042496679947, 0.05312084993359894, 0.10624169986719788, 0.024)

In [18]:
# Get the bulk weight in kg per cu.m of each component
Wb_cement = 40
Wb_sand = Vb_sand * 1680
Wb_gravel = Vb_gravel * 1525
Wb_water = Vb_water * 1000

Wb_cement, Wb_sand, Wb_gravel, Wb_water

(40, 89.24302788844622, 162.01859229747677, 24.0)

In [19]:
# Get the absolute volume (i.e. no voids) in cu.m of each component
V_cement = Wb_cement / (3.15 * 1000)
V_sand = Wb_sand / (2.65 * 1000)
V_gravel = Wb_gravel / (2.5 * 1000)
V_water = Vb_water

V_cement, V_sand, V_gravel, V_water

(0.012698412698412698, 0.03367661429752688, 0.06480743691899071, 0.024)

In [20]:
# Get the total volume of concrete per bag of cement
V_total = V_cement + V_sand + V_gravel + V_water
V_total

0.1351824639149303

In [21]:
# Calculate quantity of component in kg per cu.m of concrete
W_cement = math.ceil(Wb_cement / V_total)
W_sand = round(Wb_sand / V_total)
W_gravel = round(Wb_gravel / V_total)
W_water = round(Wb_water / V_total)
W_total = W_cement + W_sand + W_gravel + W_water

results = {"kg/m^3": [W_cement, W_gravel, W_sand, W_water, W_total]}
index = ["cement", "coarse_aggregate", "fine_aggregate", "water", "total"]

results_df = pd.DataFrame(results, index=index)
results_df

,kg/m^3
cement,296
coarse_aggregate,1199
fine_aggregate,660
water,178
total,2333


In [22]:
# Input the mix to predict strength

test_input = {
    "cement": 296,
    "blast_furnace_slag": 0,
    "fly_ash": 0,
    "silica_fume": 0,
    "coarse_aggregate": 1199,
    "fine_aggregate": 660,
    "water": 178,
    "superplasticizer": 0,
    "age": 28
}

csp_model_pipeline.predict(test_input)[0]

np.float32(36.26003)

Remarks:

- If we put some confidence in the predicted strength, it shows that the common nominal mix proportions tend to be conservative given the amount of water added is strictly monitored.
- It also shows that the mix can be optimize to save cost while still achieving the target strength.